In [24]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder,MinMaxScaler,OrdinalEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [25]:
df=pd.read_csv("titanic_data_updated (1).csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
440,441,yes,second,"Hart, Mrs. Benjamin (Esther Ada Bloomfield)",female,45.0,1,1,F.C.C. 13529,26.2500,NaN,S
857,858,yes,first,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S
785,786,no,third,"Harmer, Mr. Abraham (David Lishin)",male,25.0,0,0,374887,7.2500,NaN,S
205,206,no,third,"Strom, Miss. Telma Matilda",female,2.0,0,1,347054,10.4625,G6,S
310,311,yes,first,"Hays, Miss. Margaret Bechstein",female,24.0,0,0,11767,83.1583,C54,C


In [26]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [27]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Family_Size']=df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
790,791,no,third,"Keane, Mr. Andrew ""Andy""",male,NaN,0,0,12460,7.75,NaN,Q,Missing,n
649,650,yes,third,"Stanley, Miss. Amy Zillah Elsie",female,23.0,0,0,CA. 2314,7.55,NaN,S,Missing,n
413,414,no,second,"Cunningham, Mr. Alfred Fleming",male,NaN,0,0,239853,0.00,NaN,S,Missing,n
854,855,no,second,"Carter, Mrs. Ernest Courtenay (Lilian Hughes)",female,44.0,1,0,244252,26.00,NaN,S,Missing,n
718,719,no,third,"McEvoy, Mr. Michael",male,NaN,0,0,36568,15.50,NaN,Q,Missing,n


In [28]:
df['Deck'].value_counts()

Deck
n    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [29]:
x=df.drop('Survived',axis=1)
y=df['Survived']

In [30]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [31]:
x_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
331,332,first,"Partner, Mr. Austen",male,45.5,0,0,113043,28.5000,C124,S,C124,C
733,734,second,"Berriman, Mr. William John",male,23.0,0,0,28425,13.0000,NaN,S,Missing,n
382,383,third,"Tikkanen, Mr. Juho",male,32.0,0,0,STON/O 2. 3101293,7.9250,NaN,S,Missing,n
704,705,third,"Hansen, Mr. Henrik Juul",male,26.0,1,0,350025,7.8542,NaN,S,Missing,n
813,814,third,"Andersson, Miss. Ebba Iris Alfrida",female,6.0,4,2,347082,31.2750,NaN,S,Missing,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,107,third,"Salkjelsvik, Miss. Anna Kristine",female,21.0,0,0,343120,7.6500,NaN,S,Missing,n
270,271,first,"Cairns, Mr. Alexander",male,NaN,0,0,113798,31.0000,NaN,S,Missing,n
860,861,third,"Hansen, Mr. Claus Peter",male,41.0,2,0,350026,14.1083,NaN,S,Missing,n
435,436,first,"Carter, Miss. Lucile Polk",female,14.0,1,2,113760,120.0000,B96 B98,S,B96 B98,B


In [32]:
mean_age=x_train['Age'].mean()
std_age=x_train['Age'].std()

x_train['Z_score']=(x_train['Age']-mean_age)/std_age
musk=(abs(x_train['Z_score'])>3)

x_train=x_train[musk]
y_train=y_train[musk]

fare_Q1=x_train['Fare'].quantile(0.25)
fare_Q3=x_train['Fare'].quantile(0.75)


IQR=fare_Q3-fare_Q1

minimum=max(0, fare_Q1-1.5*IQR)
maximum=fare_Q3+1.5*IQR

x_train['Fare']=x_train['Fare'].clip(minimum,maximum)

C:\Users\user\AppData\Local\Temp\ipykernel_28408\201335385.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_train['Fare']=x_train['Fare'].clip(minimum,maximum)


In [33]:
p1=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])
p2=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('scaler',MinMaxScaler())
])
p3=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))

])
categorical_cols=[['third','second','first']]

p4=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OrdinalEncoder(categories=categorical_cols)),
    ('scaler',MinMaxScaler())
])
preprocessor=ColumnTransformer(transformers=[
    ('pipeline1',p1,['Age']),
    ('pipeline2',p2,['Fare','Family_Size']),
    ('pipeline3',p3,['Sex']),
    ('pipeline4',p4,['Pclass'])
])
preprocessor

,transformers,"[('pipeline1', ...), ('pipeline2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [34]:
le=LabelEncoder()
y_train=le.fit_transform(y_train)
y_test=le.fit_transform(y_test)

In [35]:
lr_model=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',LogisticRegression(class_weight='balanced',max_iter=1000))
    ]
)
lr_model

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline1', ...), ('pipeline2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [37]:
lr_model.fit(x_train,y_train)

lr_model['model'].coef_
lr_model['model'].intercept_
lr_model['model'].classes_

ValueError: could not convert string to float: 'A23'

In [38]:
x_test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
709,710,third,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,1,1,2661,15.2458,NaN,C,Missing,n
439,440,second,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,0,0,C.A. 18723,10.5000,NaN,S,Missing,n
840,841,third,"Alhomaki, Mr. Ilmari Rudolf",male,20.0,0,0,SOTON/O2 3101287,7.9250,NaN,S,Missing,n
720,721,second,"Harper, Miss. Annie Jessie ""Nina""",female,6.0,0,1,248727,33.0000,NaN,S,Missing,n
39,40,third,"Nicola-Yarred, Miss. Jamila",female,14.0,1,0,2651,11.2417,NaN,C,Missing,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...
433,434,third,"Kallio, Mr. Nikolai Erland",male,17.0,0,0,STON/O 2. 3101274,7.1250,NaN,S,Missing,n
773,774,third,"Elias, Mr. Dibo",male,NaN,0,0,2674,7.2250,NaN,C,Missing,n
25,26,third,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",female,38.0,1,5,347077,31.3875,NaN,S,Missing,n
84,85,second,"Ilett, Miss. Bertha",female,17.0,0,0,SO/C 14885,10.5000,NaN,S,Missing,n


In [39]:
y_pred=lr_model.predict(x_test)
lr_model.predict_proba(x_test)

AttributeError: 'ColumnTransformer' object has no attribute 'transformers_'

In [40]:
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)
precision=accuracy_score(y_test,y_pred)
print(precision)
recall=accuracy_score(y_test,y_pred)
print(recall)

NameError: name 'y_pred' is not defined